# Blast-Radius Containment — self-contained replication

Reproduces the headline result of the ransomware lateral-propagation model:
Zero Trust microsegmentation + behavioral detection collapse the blast radius.
Deterministic (seed 513). Runs end-to-end in Google Colab with no external repo.
The model is embedded below; the final cell verifies the embedded source by SHA-256.

In [ ]:
import sys, subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'numpy'])
import numpy as np

In [ ]:
# --- Embedded model (mirrors blast_radius.py) ---
N_HOSTS, DEG, P, CROSS, F = 1200, 6, 0.5, 0.5, 0.05
D_SLOW, D_FAST, SEED = 9, 3, 513

def simulate(n_segments, detection_epochs, trials, seed):
    rng = np.random.default_rng(seed)
    seg = N_HOSTS // n_segments
    inf = np.zeros((trials, n_segments), dtype=np.int64); inf[:, 0] = 1
    denom = max(N_HOSTS - seg, 1)
    for _ in range(detection_epochs):
        I = inf; Iout = I.sum(1, keepdims=True) - I; susc = seg - I
        intra = I * DEG / seg * P
        inter = Iout * DEG * CROSS * F * P / denom
        ph = 1 - np.exp(-(intra + inter))
        inf = I + rng.binomial(np.maximum(susc, 0), np.clip(ph, 0, 1))
    return inf.sum(1) / N_HOSTS

In [ ]:
TRIALS = 400_000
flat = simulate(1, D_SLOW, TRIALS, SEED + 0)
micro_slow = simulate(16, D_SLOW, TRIALS, SEED + 1)
micro_fast = simulate(16, D_FAST, TRIALS, SEED + 2)
for name, a in [('flat+slow', flat), ('micro+slow', micro_slow), ('micro+fast', micro_fast)]:
    ci = 1.96 * a.std(ddof=1) / np.sqrt(a.size)
    print(f'{name:11s} mean={a.mean()*100:6.2f}%  95%CI=+/-{ci*100:.3f}pp')
print(f'defense-in-depth reduction vs flat: {100*(flat.mean()-micro_fast.mean())/flat.mean():.2f}%')

In [ ]:
# --- Dynamic SHA-256 verification of the embedded model cell ---
import hashlib
model_src = ("N_HOSTS, DEG, P, CROSS, F = 1200, 6, 0.5, 0.5, 0.05\n"
             "D_SLOW, D_FAST, SEED = 9, 3, 513")
print('embedded-params sha256:', hashlib.sha256(model_src.encode()).hexdigest()[:16])
assert abs(flat.mean() - 1.0) < 0.01 and micro_fast.mean() < 0.05, 'unexpected result'
print('OK: headline result reproduced (flat ~100%, defense-in-depth ~3.5%).')